# Eligibility-Criteria Preprocessing and Feature Extraction

This notebook preprocesses eligibility-criteria text from the finalized
ClinicalTrials.gov analytic cohort and derives the text- and rule-based
features used in the study.

This stage does not introduce additional study exclusions.

In [1]:
import json
import re
import unicodedata
from functools import lru_cache
from pathlib import Path

import pandas as pd
import pyphen

In [2]:
ROOT_DIR = Path("ctg_full_2005_2025")

DISEASE_GROUPING_DIR = ROOT_DIR / "disease_grouping"
INPUT_FILE = (
    DISEASE_GROUPING_DIR / "disease_analysis_cohort.csv"
)

OUTPUT_DIR = ROOT_DIR / "preprocessing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
if not INPUT_FILE.exists():
    raise RuntimeError(
        f"Disease-analysis cohort file not found: {INPUT_FILE}"
    )

In [4]:
cohort_df = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
)

domain_indicator_columns = [
    "domain_neoplasms",
    "domain_nervous_system_diseases",
    "domain_cardiovascular_diseases",
    "domain_metabolic_diseases",
    "domain_respiratory_tract_diseases",
    "domain_infections",
]

required_columns = [
    "nct_id",
    "source_page",
    "study_start_date",
    "study_start_year",
    "first_posted_date",
    "study_status",
    "study_type",
    "phase",
    "conditions",
    "disease_domains",
    "n_disease_domains",
    "disease_classification_status",
    *domain_indicator_columns,
    "lead_sponsor",
    "sponsor_class",
    "enrollment",
    "sex",
    "minimum_age",
    "maximum_age",
    "countries",
    "n_countries",
    "locations",
    "eligibility_text",
]

missing_columns = [
    column
    for column in required_columns
    if column not in cohort_df.columns
]

if missing_columns:
    raise RuntimeError(
        "Required columns are missing from the disease-analysis cohort: "
        f"{missing_columns}"
    )


if cohort_df["nct_id"].isna().any():
    raise RuntimeError(
        "Missing NCT IDs found in the preprocessing input."
    )

if cohort_df["nct_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate NCT IDs found in the preprocessing input."
    )

if cohort_df["eligibility_text"].isna().any():
    raise RuntimeError(
        "Missing eligibility text found in the disease-analysis cohort."
    )


# Every preprocessing-input study must belong to at least one
# selected disease domain.
if (
    cohort_df["n_disease_domains"] < 1
).any():
    raise RuntimeError(
        "Preprocessing input contains studies with no "
        "selected disease-domain membership."
    )


# The disease-analysis cohort may contain both single-domain
# and multiple-domain studies.
valid_classification_statuses = {
    "single_domain",
    "multiple_domains",
}

if not cohort_df[
    "disease_classification_status"
].isin(valid_classification_statuses).all():
    raise RuntimeError(
        "Preprocessing input contains an unexpected "
        "disease-classification status."
    )


# Domain indicators must reconcile with the recorded number
# of disease-domain memberships.
indicator_domain_count = (
    cohort_df[domain_indicator_columns]
    .sum(axis=1)
)

if not (
    indicator_domain_count
    == cohort_df["n_disease_domains"]
).all():
    raise RuntimeError(
        "Disease-domain indicators do not reconcile with "
        "n_disease_domains in the preprocessing input."
    )


print(
    f"Disease-analysis cohort loaded: "
    f"{len(cohort_df)} studies"
)
print(
    "Studies with single-domain membership: "
    f"{int((cohort_df['n_disease_domains'] == 1).sum())}"
)
print(
    "Studies with multiple-domain membership: "
    f"{int((cohort_df['n_disease_domains'] > 1).sum())}"
)
print("Preprocessing input integrity checks passed.")

Disease-analysis cohort loaded: 197671 studies
Studies with single-domain membership: 153677
Studies with multiple-domain membership: 43994
Preprocessing input integrity checks passed.


In [5]:
def normalize_text(text):
    """
    Apply minimal normalization while preserving substantive
    eligibility information, punctuation, numbers, units,
    mathematical operators, and line structure.
    """

    if not isinstance(text, str):
        return ""

    # Standardize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Convert tabs to spaces while preserving indentation
    text = text.replace("\t", "    ")

    # Normalize compatible Unicode forms
    text = unicodedata.normalize("NFC", text)

    # Normalize escaped markup found in ClinicalTrials.gov text
    replacements = {
        r"\>": ">",
        r"\<": "<",
        r"\[": "[",
        r"\]": "]",
        r"\&": "&",
        r"\-": "-",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Remove trailing spaces from lines
    text = re.sub(
        r"[ \u00A0]+$",
        "",
        text,
        flags=re.MULTILINE,
    )

    # Collapse excessive empty lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [6]:
def normalize_section_kind(value):
    """Normalize inclusion/exclusion labels to plain ASCII keys."""

    value = unicodedata.normalize("NFKD", value)

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(char)
    )

    return value.casefold()


def normalize_heading_key(line):
    """Create a normalized comparison form for heading detection only."""

    value = unicodedata.normalize("NFKD", line)

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(char)
    )

    value = value.casefold().strip()
    value = re.sub(r"^[*•-]+[ \t]*", "", value)
    value = re.sub(r"[ \t]+", " ", value)

    return value.strip()

In [7]:
PLAIN_SECTION_HEADING_RE = re.compile(
    r"""
    ^
    (?P<kind>inclusion|exclusion)
    [ \t]+criteria
    [ \t]*:?
    $
    """,
    flags=re.VERBOSE,
)


COLON_SECTION_HEADING_RE = re.compile(
    r"""
    ^
    (?:[^:;]{1,80}[ \t]+)?
    (?P<kind>inclusion|exclusion)
    [ \t]+criteria
    (?:[ \t]+[^:;]{1,100})?
    [ \t]*:
    $
    """,
    flags=re.VERBOSE,
)


SAFE_NO_COLON_DECORATED_HEADING_RE = re.compile(
    r"""
    ^
    (?:(?:[a-z0-9][a-z0-9_/-]*[ \t]+){1,3})?
    (?P<kind>inclusion|exclusion)
    [ \t]+criteria
    (?:
        [ \t]+\([^)]{1,80}\)
        |
        [ \t]+wp[0-9]+
        |
        [ \t]+for[ \t]+part[ \t]+[a-z0-9_-]+
    )?
    $
    """,
    flags=re.VERBOSE,
)

def classify_section_heading_line(line):
    """Return inclusion/exclusion only for a complete recognized heading line."""

    if not isinstance(line, str) or not line.strip():
        return None

    key = normalize_heading_key(line)

    for pattern in (
        PLAIN_SECTION_HEADING_RE,
        COLON_SECTION_HEADING_RE,
        SAFE_NO_COLON_DECORATED_HEADING_RE,
    ):
        match = pattern.fullmatch(key)

        if match:
            return normalize_section_kind(
                match.group("kind")
            )

    return None

In [8]:
def normalize_inline_section_headings(text):
    """
    Move substantive text following a recognized heading colon
    onto the next line without rewriting arbitrary eligibility text.
    """

    if not isinstance(text, str) or not text:
        return text

    output_lines = []

    for line in text.splitlines():
        if ":" not in line:
            output_lines.append(line)
            continue

        before_colon, after_colon = line.split(":", 1)
        heading_candidate = before_colon.strip() + ":"

        if (
            after_colon.strip()
            and classify_section_heading_line(heading_candidate)
        ):
            output_lines.append(heading_candidate)
            output_lines.append(after_colon.strip())

        else:
            output_lines.append(line)

    return "\n".join(output_lines)

In [9]:
def split_eligibility_sections(text):
    """
    Identify recognized inclusion and exclusion sections line by line.

    Text outside recognized sections is not used for section-specific
    variables but remains available for whole-field analysis.

    section_split_success is True when at least one recognized
    inclusion or exclusion section is identified.
    """

    sections = {
        "inclusion": [],
        "exclusion": [],
    }

    current_kind = None
    has_inclusion = False
    has_exclusion = False

    for line in text.splitlines():
        kind = classify_section_heading_line(line)

        if kind is not None:
            current_kind = kind

            if kind == "inclusion":
                has_inclusion = True
            else:
                has_exclusion = True

            continue

        if current_kind is not None:
            sections[current_kind].append(line)

    return {
        "inclusion_text": "\n".join(
            sections["inclusion"]
        ).strip(),
        "exclusion_text": "\n".join(
            sections["exclusion"]
        ).strip(),
        "has_inclusion_section": has_inclusion,
        "has_exclusion_section": has_exclusion,
        "section_split_success": bool(
            has_inclusion or has_exclusion
        ),
    }

In [10]:
def remove_section_heading_lines(text):
    """Remove only complete physical lines recognized as section headings."""

    if not isinstance(text, str):
        return ""

    return "\n".join(
        line
        for line in text.splitlines()
        if classify_section_heading_line(line) is None
    )

In [11]:
BOILERPLATE_TRIGGER_RE = re.compile(
    r"""
    \b
    (?:meet|meets|meeting)
    [ \t]+
    (?:all|each|any|one|one[ \t]+or[ \t]+more|any[ \t]+one)
    [ \t]+of[ \t]+the[ \t]+following
    \b
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

def strip_generic_boilerplate_prefix(line):
    """
    Remove a narrowly recognized generic introductory construction
    without discarding substantive eligibility text.

    Boilerplate is removed only when text preceding the trigger is
    clearly generic. Otherwise, the original line is preserved.
    """

    if not isinstance(line, str) or not line.strip():
        return line, False

    original = line.strip()

    test_line = re.sub(
        r"^(?:[*•-]|\d{1,3}[.)])[ \t]*",
        "",
        original,
    )

    trigger = BOILERPLATE_TRIGGER_RE.search(test_line)

    if not trigger:
        return line, False

    prefix = test_line[:trigger.start()].strip()

    generic_prefix_re = re.compile(
        r"""
        ^
        (?:
            participants?
            |
            patients?
            |
            subjects?
            |
            individuals?
            |
            candidates?
            |
            participants?[ \t]+must
            |
            patients?[ \t]+must
            |
            subjects?[ \t]+must
            |
            individuals?[ \t]+must
            |
            candidates?[ \t]+must
            |
            eligible[ \t]+participants?
            |
            eligible[ \t]+patients?
            |
            eligible[ \t]+subjects?
            |
            to[ \t]+be[ \t]+eligible[,]?[ \t]*
            (?:
                participants?
                |
                patients?
                |
                subjects?
                |
                individuals?
            )?
            (?:[ \t]+must)?
        )
        [ \t]*
        $
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    # Preserve the entire original line if anything before the trigger
    # may contain substantive eligibility information.
    if prefix and not generic_prefix_re.fullmatch(prefix):
        return line, False

    colon_pos = test_line.find(":", trigger.end())

    if colon_pos != -1:
        remainder = test_line[colon_pos + 1:].strip()
        return remainder, True

    remainder = test_line[trigger.end():].strip()

    structural_tail_re = re.compile(
        r"""
        ^
        (?:
            criteria
            |
            criterion
            |
            requirements?
            |
            conditions?
        )?
        [\s.:;-]*
        $
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    if structural_tail_re.fullmatch(remainder):
        return "", True

    # Preserve the original line when substantive text may follow
    # the recognized introductory construction.
    return line, False

def remove_section_initial_boilerplate(text):
    """
    Apply boilerplate trimming only to the first non-empty content
    line after each recognized inclusion or exclusion section heading.
    """

    if not isinstance(text, str) or not text:
        return text, False, False

    output_lines = []
    current_kind = None
    waiting_for_first_content = False

    removed = {
        "inclusion": False,
        "exclusion": False,
    }

    attempted = {
        "inclusion": False,
        "exclusion": False,
    }

    for line in text.splitlines():
        kind = classify_section_heading_line(line)

        if kind is not None:
            current_kind = kind

            # Attempt boilerplate removal only once for the first
            # recognized section of each type.
            waiting_for_first_content = not attempted[kind]

            output_lines.append(line)
            continue

        if (
            waiting_for_first_content
            and current_kind is not None
            and line.strip()
        ):
            attempted[current_kind] = True

            cleaned_line, was_removed = (
                strip_generic_boilerplate_prefix(line)
            )

            if was_removed:
                removed[current_kind] = True

            if cleaned_line.strip():
                output_lines.append(cleaned_line)

            waiting_for_first_content = False
            continue

        output_lines.append(line)

    return (
        "\n".join(output_lines),
        removed["inclusion"],
        removed["exclusion"],
    )

In [12]:
LIST_MARKER_RE = re.compile(
    r"""
    ^
    (?P<indent>[ \t]*)
    (?P<marker>
        [*•-]
        |
        \d{1,3}[.)]
        |
        [A-Za-z][.)]
        |
        [ivxlcdmIVXLCDM]{1,6}[.)]
    )
    \s+
    (?P<content>.*)
    $
    """,
    flags=re.VERBOSE,
)

def classify_list_marker(marker):
    """
    Classify an explicit list marker.

    Bullet and numeric markers are unambiguous.
    Letter-like markers are retained as ambiguous and are
    interpreted from their sequence within the surrounding block.
    """

    if marker in {"*", "•", "-"}:
        return "bullet"

    if re.fullmatch(r"\d{1,3}[.)]", marker):
        return "numeric"

    marker_body = marker[:-1]

    if re.fullmatch(r"[A-Za-z]+", marker_body):
        return "ambiguous"

    return None

def roman_to_int(value):
    """Convert a Roman numeral to an integer, or return None if invalid."""

    roman_values = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    value = value.upper()

    if not re.fullmatch(
        r"M{0,3}(CM|CD|D?C{0,3})"
        r"(XC|XL|L?X{0,3})"
        r"(IX|IV|V?I{0,3})",
        value,
    ):
        return None

    total = 0
    previous = 0

    for char in reversed(value):
        current = roman_values[char]

        if current < previous:
            total -= current
        else:
            total += current
            previous = current

    return total


def is_valid_ambiguous_sequence(markers):
    """
    Return True when ambiguous letter-like markers form a plausible
    alphabetic or Roman-numeral sequence.
    """

    bodies = [
        marker[:-1]
        for marker in markers
    ]

    # Alphabetic sequence: A, B, C ... or a, b, c ...
    if all(len(body) == 1 for body in bodies):
        values = [
            ord(body.casefold()) - ord("a") + 1
            for body in bodies
        ]

        if all(
            later == earlier + 1
            for earlier, later in zip(
                values,
                values[1:],
            )
        ):
            return True

    # Roman sequence: I, II, III ... or i, ii, iii ...
    roman_values = [
        roman_to_int(body)
        for body in bodies
    ]

    if (
        all(value is not None for value in roman_values)
        and all(
            later == earlier + 1
            for earlier, later in zip(
                roman_values,
                roman_values[1:],
            )
        )
    ):
        return True

    return False


def count_top_level_listed_items(section_text):
    """
    Count explicitly represented top-level list items.

    Nested items with greater indentation are retained as part
    of the parent and are not counted separately.

    Bullet and numeric markers are accepted directly.
    Letter-like markers are accepted only when they form a plausible
    alphabetic or Roman-numeral sequence at the same indentation level.

    This measures registry list structure, not semantic criteria.
    """

    detected_items = []

    for line in section_text.splitlines():
        match = LIST_MARKER_RE.match(line)

        if not match:
            continue

        content = match.group("content").strip()

        if not content:
            continue

        indent = match.group("indent").replace("\t", "    ")
        indent_level = len(indent)

        marker = match.group("marker")
        marker_type = classify_list_marker(marker)

        if marker_type is None:
            continue

        detected_items.append(
            {
                "indent": indent_level,
                "marker": marker,
                "marker_type": marker_type,
                "content": content,
            }
        )

    if not detected_items:
        return None, False

    accepted_items = [
        item
        for item in detected_items
        if item["marker_type"] in {"bullet", "numeric"}
    ]

    ambiguous_groups = {}

    for item in detected_items:
        if item["marker_type"] != "ambiguous":
            continue

        ambiguous_groups.setdefault(
            item["indent"],
            [],
        ).append(item)

    for group in ambiguous_groups.values():
        markers = [
            item["marker"]
            for item in group
        ]

        if (
            len(group) >= 2
            and is_valid_ambiguous_sequence(markers)
        ):
            accepted_items.extend(group)

    if not accepted_items:
        return None, False

    top_indent = min(
        item["indent"]
        for item in accepted_items
    )

    top_level_items = [
        item
        for item in accepted_items
        if item["indent"] == top_indent
    ]

    return len(top_level_items), True

def count_whole_field_listed_items(text):
    """
    Count explicitly represented top-level listed items across the
    complete eligibility field.

    Recognized inclusion/exclusion heading lines are used only as
    structural boundaries so that independently formatted list blocks
    can establish their own top-level indentation.

    All eligibility text is retained for counting, including text
    outside recognized inclusion/exclusion sections.
    """

    if not isinstance(text, str) or not text.strip():
        return None, False

    blocks = []
    current_block = []

    for line in text.splitlines():

        if classify_section_heading_line(line) is not None:

            if any(
                block_line.strip()
                for block_line in current_block
            ):
                blocks.append(
                    "\n".join(current_block)
                )

            current_block = []
            continue

        current_block.append(line)

    if any(
        block_line.strip()
        for block_line in current_block
    ):
        blocks.append(
            "\n".join(current_block)
        )

    total_count = 0
    any_list_detected = False

    for block in blocks:

        block_count, block_detected = (
            count_top_level_listed_items(
                block
            )
        )

        if block_detected:
            any_list_detected = True
            total_count += block_count

    if not any_list_detected:
        return None, False

    return total_count, True

In [13]:
def remove_list_markers(text):
    """
    Remove recognized structural list markers while retaining
    the substantive text of each line.

    Bullet and numeric markers are removed directly.
    Ambiguous letter-like markers are removed only when they form
    a plausible alphabetic or Roman-numeral sequence within the same
    structural block and indentation level.
    """

    if not isinstance(text, str):
        return ""

    lines = text.splitlines()

    # Assign each line to a structural block.
    # Recognized section headings start a new block and are preserved
    # here; they are removed later by remove_section_heading_lines().
    block_ids = []
    current_block = 0

    for line in lines:
        if classify_section_heading_line(line) is not None:
            current_block += 1

        block_ids.append(current_block)

    parsed_lines = []

    # First pass: identify candidate markers.
    for line in lines:
        match = LIST_MARKER_RE.match(line)

        if not match:
            parsed_lines.append(None)
            continue

        content = match.group("content").strip()

        if not content:
            parsed_lines.append(None)
            continue

        indent = match.group("indent").replace("\t", "    ")
        indent_level = len(indent)

        marker = match.group("marker")
        marker_type = classify_list_marker(marker)

        if marker_type is None:
            parsed_lines.append(None)
            continue

        parsed_lines.append(
            {
                "indent": indent_level,
                "marker": marker,
                "marker_type": marker_type,
                "content": content,
            }
        )

    # Determine which ambiguous marker groups form valid sequences.
    accepted_ambiguous_indices = set()

    ambiguous_groups = {}

    for index, item in enumerate(parsed_lines):
        if item is None:
            continue

        if item["marker_type"] != "ambiguous":
            continue

        key = (
            block_ids[index],
            item["indent"],
        )

        ambiguous_groups.setdefault(
            key,
            [],
        ).append((index, item))

    for group in ambiguous_groups.values():
        markers = [
            item["marker"]
            for _, item in group
        ]

        if (
            len(group) >= 2
            and is_valid_ambiguous_sequence(markers)
        ):
            for index, _ in group:
                accepted_ambiguous_indices.add(index)

    cleaned_lines = []

    # Second pass: remove only validated structural markers.
    for index, (line, item) in enumerate(
        zip(lines, parsed_lines)
    ):
        if item is None:
            cleaned_lines.append(line.strip())
            continue

        marker_type = item["marker_type"]

        if marker_type in {"bullet", "numeric"}:
            cleaned_lines.append(item["content"])
            continue

        if (
            marker_type == "ambiguous"
            and index in accepted_ambiguous_indices
        ):
            cleaned_lines.append(item["content"])
            continue

        cleaned_lines.append(line.strip())

    return "\n".join(cleaned_lines)

In [14]:
def clean_analysis_text(text):
    """
    Produce the text used for feature extraction.

    Only recognized section-heading lines and structural list markers
    are removed. Substantive eligibility text is otherwise preserved.
    """

    text = remove_list_markers(text)
    text = remove_section_heading_lines(text)

    # Collapse repeated spaces within lines.
    text = re.sub(r"[ \u00A0]+", " ", text)

    # Collapse excessive empty lines while preserving line structure.
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [15]:
WORD_RE = re.compile(
    r"""
    \d+(?:[.,]\d+)*
    |
    [A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*
    """,
    flags=re.VERBOSE,
)

def get_word_tokens(text):
    """
    Return word-like tokens used consistently for study word counts
    and readability calculations.
    """
    return WORD_RE.findall(text)

In [16]:
def get_sentence_units(clean_text):
    """
    Identify sentence-like units in semi-structured eligibility text.

    Every non-empty physical line contributes at least one sentence-like
    unit. Within a physical line, reliable inline list-item boundaries
    and conventional sentence boundaries are also recognized.

    Inline structural splitting is restricted to:
    - numbered items following a semicolon, such as
      "; 2. ...", "; 3) ...", or "; (4) ...";
    - repeated lowercase parenthetical alphabetic items, such as
      "(a) ... (b) ... (c) ...";
    - dash-delimited items with surrounding whitespace when the
      following item begins with an uppercase letter;
    - dash-delimited items immediately following a colon or period,
      such as "Criteria:- Criterion ..." or "age.- Criterion ...".

    Ordinary semicolons, numeric ranges, lexical hyphens, isolated
    parenthetical letters, and dash-delimited parenthetical asides are
    not treated as sentence boundaries.
    """

    sentence_units = []

    for line in clean_text.splitlines():

        line = line.strip()

        if not line:
            continue

        protected_line = (
            line
            .replace("e.g.", "e§g§")
            .replace("E.g.", "E§g§")
            .replace("i.e.", "i§e§")
            .replace("I.e.", "I§e§")
        )

        # Activate lowercase alphabetic-list splitting only when
        # at least two such markers occur in the same physical line.
        alphabetic_markers = re.findall(
            r"\([a-z]\)",
            protected_line,
        )

        structural_pattern_parts = [
            r"""
            ;\s*
            (?=
                (?:
                    \(\d{1,3}\)
                    |
                    \d{1,3}[.)]
                )
                \s*
                [A-Za-z]
            )
            """,
            r"""
            \s+-\s+
            (?=[A-Z])
            """,
            r"""
            (?<=[:.])-\s*
            (?=[A-Z0-9])
            """,
        ]

        if len(alphabetic_markers) >= 2:
            structural_pattern_parts.append(
                r"""
                \s+
                (?=
                    \([a-z]\)
                    \s+
                )
                """
            )

        structural_pattern = "|".join(
            structural_pattern_parts
        )

        structural_parts = re.split(
            structural_pattern,
            protected_line,
            flags=re.VERBOSE,
        )

        for structural_part in structural_parts:

            structural_part = structural_part.strip()

            if not structural_part:
                continue

            parts = re.split(
                r"""
                (?<=[!?])\s+
                |
                (?<=[A-Za-z)\]])\.\s+(?=[A-Z0-9])
                """,
                structural_part,
                flags=re.VERBOSE,
            )

            for part in parts:

                part = (
                    part
                    .replace("e§g§", "e.g.")
                    .replace("E§g§", "E.g.")
                    .replace("i§e§", "i.e.")
                    .replace("I§e§", "I.e.")
                    .strip()
                )

                if get_word_tokens(part):
                    sentence_units.append(part)

    return sentence_units

In [17]:
hyphenator = pyphen.Pyphen(lang="en_US")


@lru_cache(maxsize=200000)
def _count_syllables_clean_word(clean_word):
    """
    Cached Pyphen syllable estimate for a normalized alphabetic word.
    Caching changes performance only, not the calculation.
    """

    if not clean_word:
        return 0

    hyphenated = hyphenator.inserted(clean_word)

    return max(
        1,
        hyphenated.count("-") + 1,
    )


def count_syllables_in_word(word):
    """
    Estimate English syllables for alphabetic lexical tokens.

    Tokens containing digits or no alphabetic characters are excluded
    from syllable estimation because their spoken realization is not
    reliably represented by orthographic syllabification.
    """

    if not re.fullmatch(
        r"[A-Za-z]+(?:[-'][A-Za-z]+)*",
        word,
    ):
        return 0

    clean_word = re.sub(
        r"[^A-Za-z]",
        "",
        word,
    ).lower()

    return _count_syllables_clean_word(clean_word)

In [18]:
def get_readability_word_tokens(text):
    """
    Return lexical alphabetic tokens used for readability calculation.

    Word-like tokens containing digits are excluded as whole tokens
    rather than fragmented into alphabetic substrings, because their
    spoken realization is not reliably represented by English
    orthographic syllabification.
    """

    tokens = get_word_tokens(text)

    return [
        token
        for token in tokens
        if re.fullmatch(
            r"[A-Za-z]+(?:[-'][A-Za-z]+)*",
            token,
        )
    ]


def calculate_flesch_reading_ease(text):
    """
    Calculate Flesch Reading Ease using lexical word tokens and the
    sentence-like units defined for semi-structured eligibility text.

    Higher scores indicate easier-to-read text.
    """

    words = get_readability_word_tokens(text)
    sentence_units = get_sentence_units(text)

    n_words = len(words)
    n_sentence_units = len(sentence_units)

    if n_words == 0 or n_sentence_units == 0:
        return None

    n_syllables = sum(
        count_syllables_in_word(word)
        for word in words
    )

    flesch_reading_ease = (
        206.835
        - 1.015 * (n_words / n_sentence_units)
        - 84.6 * (n_syllables / n_words)
    )

    return flesch_reading_ease

In [19]:
def compile_patterns(pattern_list):
    """Compile feature patterns using common regex flags."""

    return [
        re.compile(
            pattern,
            flags=(
                re.IGNORECASE
                | re.UNICODE
                | re.VERBOSE
            ),
        )
        for pattern in pattern_list
    ]

In [20]:
def collect_non_overlapping_matches(
    text,
    patterns,
    exclusion_patterns=None,
):
    """
    Collect non-overlapping matches within one semantic characteristic.

    Candidate matches are processed from left to right. When multiple
    candidates begin at the same position, the longest candidate is
    considered first. Candidates overlapping an already selected match
    are discarded.
    """

    candidates = []

    for pattern in patterns:
        for match in pattern.finditer(text):
            candidates.append(
                (
                    match.start(),
                    match.end(),
                    match.group(0),
                )
            )

    # Earlier positions first; at the same position, longer matches first.
    candidates.sort(
        key=lambda x: (
            x[0],
            -(x[1] - x[0]),
        )
    )

    selected = []

    for start, end, value in candidates:

        # Exclude predefined false-positive expressions.
        if exclusion_patterns:
            excluded = any(
                pattern.fullmatch(value.strip())
                for pattern in exclusion_patterns
            )

            if excluded:
                continue

        overlap = any(
            not (
                end <= previous_start
                or start >= previous_end
            )
            for (
                previous_start,
                previous_end,
                _
            ) in selected
        )

        if overlap:
            continue

        selected.append(
            (start, end, value)
        )

    selected.sort(
        key=lambda x: x[0]
    )

    return selected

In [21]:
# Numeric value used by quantitative and temporal detectors.
NUMBER = r"(?:\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:[.,]\d+)?)"


# Time units are shared with the temporal-expression detector.
TIME_UNIT = (
    r"(?:"
    r"minutes?|mins?|"
    r"hours?|hrs?|"
    r"days?|"
    r"weeks?|"
    r"months?|"
    r"years?|"
    r"decades?"
    r")"
)


# Explicit measurement units recognized by the quantitative detector.
MEASUREMENT_UNIT = (
    r"(?:"
    r"%|"
    r"kg/m(?:2|²)|"
    r"mg/dL|"
    r"g/dL|"
    r"µg/mL|"
    r"mcg/mL|"
    r"ng/mL|"
    r"mmol/L|"
    r"mol/L|"
    r"mEq/L|"
    r"mIU/mL|"
    r"IU/mL|"
    r"U/L|"
    r"cells?/mm(?:3|³)|"
    r"/mm(?:3|³)|"
    r"mmHg|"
    r"msec|"
    r"ms|"
    r"kg|"
    r"g|"
    r"mg|"
    r"µg|"
    r"mcg|"
    r"ng|"
    r"mL|"
    r"ml|"
    r"L|"
    r"mm|"
    r"cm|"
    r"m"
    r")"
)


# Multiples of the upper limit of normal.
ULN_UNIT = (
    r"(?:"
    r"(?:x|×)[ \t]*"
    r"(?:the[ \t]+)?"
    r"(?:upper[ \t]+limit[ \t]+of[ \t]+normal|ULN)"
    r")"
)


# Shared quantitative unit.
#
# The final negative look-ahead prevents short units such as
# "m", "g", or "L" from matching the beginning of ordinary words
# (for example, "5 male participants" -> "5 m").
QUANTITATIVE_UNIT = (
    rf"(?:"
    rf"{TIME_UNIT}|"
    rf"{MEASUREMENT_UNIT}|"
    rf"{ULN_UNIT}"
    rf")"
    rf"(?![A-Za-z])"
)


QUANTITATIVE_PATTERNS = compile_patterns([

    # Symbolic comparator + number
    # Examples:
    #   >=18
    #   < 5
    #   <=1.5 mg/dL

    rf"""
    (?<!\w)
    (?:>=|<=|≥|≤|>|<|=)
    [ \t]*
    {NUMBER}
    (?:[ \t]*{QUANTITATIVE_UNIT})?
    """,

    # Lexical comparator + number
    # Examples:
    #   at least 18
    #   no more than 5
    #   less than 10 years

    rf"""
    \b
    (?:
        at[ \t]+least
        |
        at[ \t]+most
        |
        greater[ \t]+than
        |
        more[ \t]+than
        |
        less[ \t]+than
        |
        no[ \t]+more[ \t]+than
        |
        no[ \t]+less[ \t]+than
        |
        not[ \t]+less[ \t]+than
        |
        not[ \t]+more[ \t]+than
        |
        no[ \t]+longer[ \t]+than
    )
    [ \t]+
    {NUMBER}
    (?:[ \t]*{QUANTITATIVE_UNIT})?
    """,

    # Between X and Y
    # Example:
    #   between 18 and 65 years

    rf"""
    \bbetween
    [ \t]+{NUMBER}
    [ \t]+and
    [ \t]+{NUMBER}
    (?:[ \t]*{QUANTITATIVE_UNIT})?
    """,

    # Explicit range
    # Examples:
    #   18-65 years
    #   18 to 65 years

    rf"""
    (?<!\w)
    {NUMBER}
    [ \t]*
    (?:-|–|—|to)
    [ \t]*
    {NUMBER}
    [ \t]*
    {QUANTITATIVE_UNIT}
    """,

    # Number with an explicit quantitative unit
    # Examples:
    #   7%
    #   30 kg/m2
    #   72 hours

    rf"""
    (?<!\w)
    {NUMBER}
    [ \t]*
    {QUANTITATIVE_UNIT}
    """
])

In [22]:
TEMPORAL_REFERENCE = (
    r"(?:"
    r"screening|"
    r"enrollment|"
    r"enrolment|"
    r"randomi[sz]ation|"
    r"baseline|"
    r"surgery|"
    r"treatment|"
    r"therapy|"
    r"diagnosis|"
    r"study[ \t]+entry|"
    r"first[ \t]+dose|"
    r"first[ \t]+patch[ \t]+application|"
    r"recruitment"
    r")"
)


TEMPORAL_PATTERNS = compile_patterns([

    # Composite time window + reference point
    # Example: within 3 months before screening
    rf"""
    \bwithin
    [ \t]+(?:the[ \t]+)?
    (?:(?:past|previous|prior|preceding|last)[ \t]+)?
    {NUMBER}
    [ \t]+{TIME_UNIT}
    [ \t]+
    (?:prior[ \t]+to|before|after|following)
    [ \t]+
    {TEMPORAL_REFERENCE}
    \b
    """,

    # Duration
    # Examples: for 6 months; for at least 2 weeks
    rf"""
    \bfor
    [ \t]+
    (?:
        (?:
            at[ \t]+least
            |
            at[ \t]+most
            |
            more[ \t]+than
            |
            less[ \t]+than
            |
            no[ \t]+more[ \t]+than
            |
            no[ \t]+less[ \t]+than
        )
        [ \t]+
    )?
    {NUMBER}
    [ \t]+
    {TIME_UNIT}
    \b
    """,

    # Comparator + duration without "for"
    # Examples: at least 2 years; more than 6 months
    rf"""
    \b
    (?:
        at[ \t]+least
        |
        at[ \t]+most
        |
        more[ \t]+than
        |
        less[ \t]+than
        |
        no[ \t]+more[ \t]+than
        |
        no[ \t]+less[ \t]+than
        |
        not[ \t]+more[ \t]+than
        |
        not[ \t]+less[ \t]+than
    )
    [ \t]+
    {NUMBER}
    [ \t]+
    {TIME_UNIT}
    \b
    """,

    # Explicit window
    # Examples: within 30 days; within the previous 3 months
    rf"""
    \bwithin
    [ \t]+(?:the[ \t]+)?
    (?:(?:past|previous|prior|preceding|last)[ \t]+)?
    {NUMBER}
    [ \t]+
    {TIME_UNIT}
    \b
    """,

    # During/in the previous...
    # Example: in the last 3 months
    rf"""
    \b(?:during|in)
    [ \t]+(?:the[ \t]+)?
    (?:past|previous|prior|preceding|last)
    [ \t]+
    {NUMBER}
    [ \t]+
    {TIME_UNIT}
    \b
    """,

    # Number/time unit followed by reference relation
    # Examples: 6 months prior to screening; 30 days before enrollment
    rf"""
    \b
    {NUMBER}
    [ \t]+
    {TIME_UNIT}
    [ \t]+
    (?:prior[ \t]+to|before|after|following)
    [ \t]+
    {TEMPORAL_REFERENCE}
    \b
    """,

    # Explicit reference-point relation
    # Examples: prior to enrollment; after surgery; since diagnosis
    rf"""
    \b
    (?:prior[ \t]+to|before|after|following|since)
    [ \t]+
    {TEMPORAL_REFERENCE}
    \b
    """,

    # At explicit trial reference point
    # Examples: at screening; at baseline
    rf"""
    \bat
    [ \t]+
    (?:
        screening|
        enrollment|
        enrolment|
        randomi[sz]ation|
        baseline|
        study[ \t]+entry
    )
    \b
    """,

    # Frequency
    # Examples: three times per week; 3 times per week; twice per day
    rf"""
    \b
    (?:
        once|
        twice|
        three[ \t]+times|
        {NUMBER}[ \t]+times
    )
    [ \t]+
    (?:per[ \t]+|a[ \t]+)?
    (?:day|week|month)
    \b
    """,

    # Every X time units
    # Example: every 2 weeks
    rf"""
    \bevery
    [ \t]+
    {NUMBER}
    [ \t]+
    {TIME_UNIT}
    \b
    """,

    # Daily/weekly/monthly for X duration
    # Example: daily for 6 months
    rf"""
    \b
    (?:daily|weekly|monthly)
    [ \t]+for
    [ \t]+
    {NUMBER}
    [ \t]+
    {TIME_UNIT}
    \b
    """
])

In [23]:
def is_temporal_age_excluded(text, start, end):
    """
    Return True when a detected temporal expression is an age
    specification rather than a temporal duration.

    Examples excluded:
        at least 18 years old
        at least 18 years of age
        more than 65 years old
        less than 18 years of age

    Genuine durations remain:
        at least 2 years postmenopausal
        more than 5 years since diagnosis
        less than 6 months before screening
    """

    following_text = text[
        end:min(len(text), end + 30)
    ]

    age_context_pattern = re.compile(
        r"""
        ^\s*
        (?:
            old
            |
            of\s+age
        )
        \b
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    return bool(
        age_context_pattern.search(following_text)
    )

In [24]:
NEGATION_PATTERNS = compile_patterns([
    r"\babsence[ \t]+of\b",
    r"\bfree[ \t]+of\b",
    r"\bneither\b",
    r"\bnor\b",
    r"\bnever\b",
    r"\bwithout\b",
    r"\bnot\b",
    r"\bno\b(?!\.)",
])


def is_negation_excluded(text, start, end):
    """
    Return True when a detected 'no' or 'not' is part of a
    quantitative comparator rather than an independent negation marker.

    Examples excluded:
        no more than
        no less than
        no longer than
        not more than
        not less than
    """

    context = text[
        start:min(len(text), start + 50)
    ]

    exclusion_pattern = re.compile(
        r"""
        ^
        (?:
            no\s+more\s+than
            |
            no\s+less\s+than
            |
            no\s+longer\s+than
            |
            not\s+more\s+than
            |
            not\s+less\s+than
        )
        \b
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    return bool(
        exclusion_pattern.search(context)
    )

In [25]:
CONDITIONAL_PATTERNS = compile_patterns([
    r"\bon[ \t]+condition[ \t]+that\b",
    r"\bin[ \t]+the[ \t]+event[ \t]+that\b",
    r"\bprovided[ \t]+that\b",
    r"\bas[ \t]+long[ \t]+as\b",
    r"\bonly[ \t]+if\b",
    r"\bif\b",
])

In [26]:
EXCEPTION_PATTERNS = compile_patterns([
    r"\bwith[ \t]+the[ \t]+exception[ \t]+of\b",
    r"\bexcept[ \t]+for\b",
    r"\bother[ \t]+than\b",
    r"\bexcluding\b",
    r"\bunless\b",
    r"\bexcept\b",
])

In [27]:
SEMANTIC_CHARACTERISTIC_DEFINITIONS = {
    "quantitative": {
        "patterns": QUANTITATIVE_PATTERNS,
        "exclusions": None,
    },
    "temporal": {
        "patterns": TEMPORAL_PATTERNS,
        "exclusions": None,
    },
    "negation": {
        "patterns": NEGATION_PATTERNS,
        "exclusions": None,
    },
    "conditional": {
        "patterns": CONDITIONAL_PATTERNS,
        "exclusions": None,
    },
    "exception": {
        "patterns": EXCEPTION_PATTERNS,
        "exclusions": None,
    },
}

In [28]:
def process_study(study_row):
    """
    Preprocess one study and extract eligibility-criteria measures
    and semantic characteristics.

    The input is one row from the finalized analytic cohort.
    No study is excluded at this stage.
    """

    row = study_row.to_dict()
    raw_text = row["eligibility_text"]

    nct_id = row["nct_id"]
    warnings = []

    # Missing eligibility text at this stage indicates an upstream
    # cohort inconsistency rather than a preprocessing exclusion.
    if not isinstance(raw_text, str) or not raw_text.strip():
        raise RuntimeError(
            f"Missing eligibility text in finalized analytic cohort: {nct_id}"
        )

    row["has_eligibility_text"] = True

    # Minimal normalization
    normalized_text = normalize_text(raw_text)

    normalized_text = normalize_inline_section_headings(
        normalized_text
    )

    # Remove only recognized generic introductory boilerplate at the
    # beginning of inclusion/exclusion sections.
    (
        analysis_source_text,
        inclusion_boilerplate_removed,
        exclusion_boilerplate_removed,
    ) = remove_section_initial_boilerplate(
        normalized_text
    )

    # Split recognized inclusion/exclusion sections for
    # section-specific variables only.
    sections = split_eligibility_sections(
        analysis_source_text
    )

    inclusion_text = sections["inclusion_text"]
    exclusion_text = sections["exclusion_text"]

    row["has_inclusion_section"] = (
        sections["has_inclusion_section"]
    )

    row["has_exclusion_section"] = (
        sections["has_exclusion_section"]
    )

    row["section_split_success"] = (
        sections["section_split_success"]
    )

    if not sections["section_split_success"]:
        warnings.append("NO_SECTION_HEADING")

    row["inclusion_boilerplate_removed"] = (
        inclusion_boilerplate_removed
    )

    row["exclusion_boilerplate_removed"] = (
        exclusion_boilerplate_removed
    )

    # Section-specific listed-item counts
    if inclusion_text:
        (
            inclusion_item_count,
            _
        ) = count_top_level_listed_items(
            inclusion_text
        )
    else:
        inclusion_item_count = None

    if exclusion_text:
        (
            exclusion_item_count,
            _
        ) = count_top_level_listed_items(
            exclusion_text
        )
    else:
        exclusion_item_count = None

    # Primary listed-item count uses the complete eligibility field.
    (
        total_item_count,
        total_list_detected,
    ) = count_whole_field_listed_items(
        analysis_source_text
    )

    row["n_inclusion_listed_items"] = (
        inclusion_item_count
    )

    row["n_exclusion_listed_items"] = (
        exclusion_item_count
    )

    row["n_total_listed_items"] = (
        total_item_count
    )

    row["list_structure_detected"] = bool(
        total_list_detected
    )

    row["listed_item_count_available"] = (
        total_item_count is not None
    )

    if total_item_count is None:
        warnings.append("NO_LIST_STRUCTURE")

    # Clean representations used for feature extraction.
    #
    # Whole-field measures do not depend on successful recognition
    # of inclusion/exclusion headings.
    inclusion_clean = (
        clean_analysis_text(inclusion_text)
        if inclusion_text
        else ""
    )

    exclusion_clean = (
        clean_analysis_text(exclusion_text)
        if exclusion_text
        else ""
    )

    eligibility_clean = clean_analysis_text(
        analysis_source_text
    )

    # Word and sentence measures
    words = get_word_tokens(
        eligibility_clean
    )

    sentences = get_sentence_units(
        eligibility_clean
    )

    total_words = len(words)
    sentence_count = len(sentences)

    row["total_eligibility_word_count"] = (
        total_words
    )

    row["sentence_count"] = (
        sentence_count
    )

    row["mean_sentence_length"] = (
        total_words / sentence_count
        if sentence_count
        else None
    )

    row["flesch_reading_ease"] = (
        calculate_flesch_reading_ease(
            eligibility_clean
        )
    )

    row["inclusion_word_count"] = (
        len(get_word_tokens(inclusion_clean))
        if row["has_inclusion_section"]
        else None
    )

    row["exclusion_word_count"] = (
        len(get_word_tokens(exclusion_clean))
        if row["has_exclusion_section"]
        else None
    )

    # Rule-based semantic-characteristic counts
    for characteristic_name, definition in (
        SEMANTIC_CHARACTERISTIC_DEFINITIONS.items()
    ):

        matches = collect_non_overlapping_matches(
            text=eligibility_clean,
            patterns=definition["patterns"],
            exclusion_patterns=definition["exclusions"],
        )

        if characteristic_name == "negation":
            matches = [
                (start, end, value)
                for start, end, value in matches
                if not is_negation_excluded(
                    eligibility_clean,
                    start,
                    end,
                )
            ]

        if characteristic_name == "temporal":
            matches = [
                (start, end, value)
                for start, end, value in matches
                if not is_temporal_age_excluded(
                    eligibility_clean,
                    start,
                    end,
                )
            ]

        count = len(matches)

        row[f"{characteristic_name}_count"] = count
        row[f"{characteristic_name}_present"] = count > 0

        row[
            f"{characteristic_name}_rate_per_1000_words"
        ] = (
            count / total_words * 1000
            if total_words
            else None
        )

    row["preprocessing_warning"] = "|".join(
        warnings
    )

    return row

In [29]:
FINAL_OUTPUT_COLUMNS = [
    # Study metadata
    "nct_id",
    "source_page",
    "study_start_date",
    "study_start_year",
    "first_posted_date",
    "study_status",
    "study_type",
    "phase",
    "conditions",

    # Disease-domain classification
    "disease_domains",
    "n_disease_domains",
    "disease_classification_status",
    *domain_indicator_columns,

    "lead_sponsor",
    "sponsor_class",
    "enrollment",
    "sex",
    "minimum_age",
    "maximum_age",
    "countries",
    "n_countries",
    "locations",

    # Preprocessing / QC
    "has_eligibility_text",
    "has_inclusion_section",
    "has_exclusion_section",
    "section_split_success",
    "inclusion_boilerplate_removed",
    "exclusion_boilerplate_removed",
    "list_structure_detected",
    "listed_item_count_available",

    # Eligibility burden and linguistic measures
    "n_inclusion_listed_items",
    "n_exclusion_listed_items",
    "n_total_listed_items",
    "inclusion_word_count",
    "exclusion_word_count",
    "total_eligibility_word_count",
    "sentence_count",
    "mean_sentence_length",
    "flesch_reading_ease",

    # Semantic characteristics
    "quantitative_count",
    "quantitative_present",
    "quantitative_rate_per_1000_words",

    "temporal_count",
    "temporal_present",
    "temporal_rate_per_1000_words",

    "negation_count",
    "negation_present",
    "negation_rate_per_1000_words",

    "conditional_count",
    "conditional_present",
    "conditional_rate_per_1000_words",

    "exception_count",
    "exception_present",
    "exception_rate_per_1000_words",

    # General preprocessing warning
    "preprocessing_warning",
]

In [30]:
PROCESSED_CHUNKS_DIR = OUTPUT_DIR / "processed_chunks"
PROCESSED_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PROCESSED_FILE = (
    OUTPUT_DIR / "eligibility_features.csv"
)

PREPROCESSING_SUMMARY_FILE = (
    OUTPUT_DIR / "preprocessing_summary.json"
)

CHUNK_SIZE = 5000

In [31]:
def save_csv_atomic(dataframe, output_path):
    """Write a CSV atomically using a temporary file."""

    temp_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )

    dataframe.to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig",
    )

    temp_path.replace(output_path)


n_studies = len(cohort_df)

n_chunks = (
    n_studies + CHUNK_SIZE - 1
) // CHUNK_SIZE


for chunk_index in range(n_chunks):

    start = chunk_index * CHUNK_SIZE
    end = min(
        start + CHUNK_SIZE,
        n_studies,
    )

    chunk_number = chunk_index + 1

    chunk_file = (
        PROCESSED_CHUNKS_DIR
        / f"chunk_{chunk_number:06d}.csv"
    )

    input_chunk = cohort_df.iloc[
        start:end
    ].copy()

    expected_chunk_ids = (
        input_chunk["nct_id"]
        .astype(str)
        .tolist()
    )

    # Resume safely from a previously completed chunk.
    if chunk_file.exists():

        existing_chunk = pd.read_csv(
            chunk_file,
            encoding="utf-8-sig",
        )

        missing_columns = [
            column
            for column in FINAL_OUTPUT_COLUMNS
            if column not in existing_chunk.columns
        ]

        if missing_columns:
            raise RuntimeError(
                f"Existing chunk {chunk_file.name} is missing "
                f"required columns: {missing_columns}"
            )

        existing_ids = (
            existing_chunk["nct_id"]
            .astype(str)
            .tolist()
        )

        if existing_ids != expected_chunk_ids:
            raise RuntimeError(
                f"Existing chunk {chunk_file.name} does not match "
                "the current disease-analysis cohort input."
            )

        print(
            f"Skipping completed chunk "
            f"{chunk_number}/{n_chunks}: "
            f"{len(existing_chunk)} studies"
        )

        continue

    processed_rows = []

    for _, study_row in input_chunk.iterrows():

        nct_id = study_row["nct_id"]

        try:
            processed_row = process_study(
                study_row
            )

        except Exception as exc:
            raise RuntimeError(
                f"Preprocessing failed for study {nct_id} "
                f"in chunk {chunk_number}."
            ) from exc

        processed_rows.append(
            processed_row
        )

    processed_chunk_df = pd.DataFrame(
        processed_rows
    )

    missing_columns = [
        column
        for column in FINAL_OUTPUT_COLUMNS
        if column not in processed_chunk_df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Processed chunk {chunk_number} is missing "
            f"required columns: {missing_columns}"
        )

    processed_chunk_df = processed_chunk_df[
        FINAL_OUTPUT_COLUMNS
    ].copy()

    processed_ids = (
        processed_chunk_df["nct_id"]
        .astype(str)
        .tolist()
    )

    if processed_ids != expected_chunk_ids:
        raise RuntimeError(
            f"NCT-ID reconciliation failed within "
            f"chunk {chunk_number}."
        )

    save_csv_atomic(
        processed_chunk_df,
        chunk_file,
    )

    print(
        f"Saved chunk {chunk_number}/{n_chunks}: "
        f"{len(processed_chunk_df)} studies"
    )

Saved chunk 1/40: 5000 studies
Saved chunk 2/40: 5000 studies
Saved chunk 3/40: 5000 studies
Saved chunk 4/40: 5000 studies
Saved chunk 5/40: 5000 studies
Saved chunk 6/40: 5000 studies
Saved chunk 7/40: 5000 studies
Saved chunk 8/40: 5000 studies
Saved chunk 9/40: 5000 studies
Saved chunk 10/40: 5000 studies
Saved chunk 11/40: 5000 studies
Saved chunk 12/40: 5000 studies
Saved chunk 13/40: 5000 studies
Saved chunk 14/40: 5000 studies
Saved chunk 15/40: 5000 studies
Saved chunk 16/40: 5000 studies
Saved chunk 17/40: 5000 studies
Saved chunk 18/40: 5000 studies
Saved chunk 19/40: 5000 studies
Saved chunk 20/40: 5000 studies
Saved chunk 21/40: 5000 studies
Saved chunk 22/40: 5000 studies
Saved chunk 23/40: 5000 studies
Saved chunk 24/40: 5000 studies
Saved chunk 25/40: 5000 studies
Saved chunk 26/40: 5000 studies
Saved chunk 27/40: 5000 studies
Saved chunk 28/40: 5000 studies
Saved chunk 29/40: 5000 studies
Saved chunk 30/40: 5000 studies
Saved chunk 31/40: 5000 studies
Saved chunk 32/40

In [32]:
chunk_files = sorted(
    PROCESSED_CHUNKS_DIR.glob("chunk_*.csv")
)

if len(chunk_files) != n_chunks:
    raise RuntimeError(
        "Processed chunk count does not match the expected number of chunks: "
        f"{len(chunk_files)} found versus {n_chunks} expected."
    )


processed_parts = []

for chunk_file in chunk_files:
    chunk_df = pd.read_csv(
        chunk_file,
        encoding="utf-8-sig",
    )

    missing_columns = [
        column
        for column in FINAL_OUTPUT_COLUMNS
        if column not in chunk_df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"{chunk_file.name} is missing required columns: "
            f"{missing_columns}"
        )

    processed_parts.append(
        chunk_df[FINAL_OUTPUT_COLUMNS]
    )


processed_df = pd.concat(
    processed_parts,
    ignore_index=True,
)


input_ids = (
    cohort_df["nct_id"]
    .astype(str)
    .tolist()
)

processed_ids = (
    processed_df["nct_id"]
    .astype(str)
    .tolist()
)


if len(processed_df) != len(cohort_df):
    raise RuntimeError(
        "Processed row count does not match preprocessing input: "
        f"{len(processed_df)} processed versus "
        f"{len(cohort_df)} input studies."
    )

if processed_df["nct_id"].isna().any():
    raise RuntimeError(
        "Missing NCT IDs found in the combined preprocessing output."
    )

if processed_df["nct_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate NCT IDs found in the combined preprocessing output."
    )

if processed_ids != input_ids:
    raise RuntimeError(
        "Processed NCT IDs do not exactly match the preprocessing input "
        "in both membership and order."
    )


print()
print("Preprocessing reconciliation passed.")
print(f"Input studies: {len(cohort_df)}")
print(f"Processed studies: {len(processed_df)}")
print(
    f"Unique processed NCT IDs: "
    f"{processed_df['nct_id'].nunique()}"
)


Preprocessing reconciliation passed.
Input studies: 197671
Processed studies: 197671
Unique processed NCT IDs: 197671


In [33]:
count_columns = [
    "n_inclusion_listed_items",
    "n_exclusion_listed_items",
    "n_total_listed_items",
    "inclusion_word_count",
    "exclusion_word_count",
    "total_eligibility_word_count",
    "sentence_count",
    "quantitative_count",
    "temporal_count",
    "negation_count",
    "conditional_count",
    "exception_count",
]

semantic_characteristic_names = [
    "quantitative",
    "temporal",
    "negation",
    "conditional",
    "exception",
]


# Count variables must never be negative.
for column in count_columns:
    negative_mask = (
        processed_df[column].notna()
        & (processed_df[column] < 0)
    )

    if negative_mask.any():
        raise RuntimeError(
            f"Negative values found in {column}: "
            f"{int(negative_mask.sum())} studies."
        )


# Every retained study must have analyzable whole-field text.
if processed_df["total_eligibility_word_count"].isna().any():
    raise RuntimeError(
        "Missing total eligibility word counts found."
    )

if (
    processed_df["total_eligibility_word_count"] <= 0
).any():
    raise RuntimeError(
        "Zero or negative total eligibility word counts found."
    )

if processed_df["sentence_count"].isna().any():
    raise RuntimeError(
        "Missing sentence counts found."
    )

if (
    processed_df["sentence_count"] <= 0
).any():
    raise RuntimeError(
        "Zero or negative sentence counts found."
    )


# Mean sentence length must equal words / sentence units.
expected_mean_sentence_length = (
    processed_df["total_eligibility_word_count"]
    / processed_df["sentence_count"]
)

mean_sentence_difference = (
    processed_df["mean_sentence_length"]
    - expected_mean_sentence_length
).abs()

if (
    mean_sentence_difference > 1e-10
).any():
    raise RuntimeError(
        "Mean sentence length reconciliation failed."
    )

# Flesch Reading Ease should be available whenever
# readability words and sentence units exist.
if processed_df["flesch_reading_ease"].isna().any():
    raise RuntimeError(
        "Missing Flesch Reading Ease values found."
    )

# Listed-item availability must agree with the listed-item count.
listed_available_expected = (
    processed_df["n_total_listed_items"].notna()
)

if not (
    processed_df["listed_item_count_available"]
    == listed_available_expected
).all():
    raise RuntimeError(
        "Listed-item availability is inconsistent with "
        "n_total_listed_items."
    )


# Semantic-characteristic counts, presence indicators, and rates must agree.
for characteristic_name in semantic_characteristic_names:

    count_column = f"{characteristic_name}_count"
    present_column = f"{characteristic_name}_present"
    rate_column = (
        f"{characteristic_name}_rate_per_1000_words"
    )

    if processed_df[count_column].isna().any():
        raise RuntimeError(
            f"Missing values found in {count_column}."
        )

    expected_present = (
        processed_df[count_column] > 0
    )

    if not (
        processed_df[present_column]
        == expected_present
    ).all():
        raise RuntimeError(
            f"{present_column} is inconsistent "
            f"with {count_column}."
        )

    expected_rate = (
        processed_df[count_column]
        / processed_df["total_eligibility_word_count"]
        * 1000
    )

    rate_difference = (
        processed_df[rate_column]
        - expected_rate
    ).abs()

    if (
        rate_difference > 1e-10
    ).any():
        raise RuntimeError(
            f"{rate_column} reconciliation failed."
        )


# Every processed study must retain at least one selected
# disease-domain membership.
if (
    processed_df["n_disease_domains"] < 1
).any():
    raise RuntimeError(
        "Processed dataset contains studies with no "
        "selected disease-domain membership."
    )


# Disease-domain indicators must reconcile with
# n_disease_domains.
indicator_domain_count = (
    processed_df[domain_indicator_columns]
    .sum(axis=1)
)

if not (
    indicator_domain_count
    == processed_df["n_disease_domains"]
).all():
    raise RuntimeError(
        "Disease-domain indicators do not reconcile with "
        "n_disease_domains in the preprocessing output."
    )


# Classification status must remain valid.
valid_classification_statuses = {
    "single_domain",
    "multiple_domains",
}

if not processed_df[
    "disease_classification_status"
].isin(valid_classification_statuses).all():
    raise RuntimeError(
        "Processed dataset contains an unexpected "
        "disease-classification status."
    )

print("Final preprocessing variable-level QC passed.")
print(
    "Studies with recognized inclusion section: "
    f"{int(processed_df['has_inclusion_section'].sum())}"
)
print(
    "Studies with recognized exclusion section: "
    f"{int(processed_df['has_exclusion_section'].sum())}"
)
print(
    "Studies with no recognized section heading: "
    f"{int((~processed_df['section_split_success']).sum())}"
)
print(
    "Studies with listed-item count available: "
    f"{int(processed_df['listed_item_count_available'].sum())}"
)
print(
    "Studies with single-domain membership: "
    f"{int((processed_df['n_disease_domains'] == 1).sum())}"
)

print(
    "Studies with multiple-domain membership: "
    f"{int((processed_df['n_disease_domains'] > 1).sum())}"
)

print(
    "Disease domains represented: "
    f"{sum(processed_df[column].any() for column in domain_indicator_columns)}"
)

Final preprocessing variable-level QC passed.
Studies with recognized inclusion section: 193899
Studies with recognized exclusion section: 192196
Studies with no recognized section heading: 3029
Studies with listed-item count available: 194776
Studies with single-domain membership: 153677
Studies with multiple-domain membership: 43994
Disease domains represented: 6


In [34]:
warning_counts = (
    processed_df["preprocessing_warning"]
    .fillna("")
    .value_counts(dropna=False)
    .rename_axis("preprocessing_warning")
    .reset_index(name="n_studies")
)

missingness_columns = [
    "n_inclusion_listed_items",
    "n_exclusion_listed_items",
    "n_total_listed_items",
    "inclusion_word_count",
    "exclusion_word_count",
    "mean_sentence_length",
    "flesch_reading_ease",
]

missingness_summary = pd.DataFrame({
    "variable": missingness_columns,
    "n_missing": [
        int(processed_df[column].isna().sum())
        for column in missingness_columns
    ],
})

missingness_summary["percent_missing"] = (
    missingness_summary["n_missing"]
    / len(processed_df)
    * 100
)


print("Preprocessing warnings")
print(
    warning_counts.to_string(index=False)
)

print()
print("Missingness in derived variables")
print(
    missingness_summary.to_string(
        index=False,
        formatters={
            "percent_missing": "{:.2f}".format
        },
    )
)

Preprocessing warnings
               preprocessing_warning  n_studies
                                         191906
                  NO_SECTION_HEADING       2870
                   NO_LIST_STRUCTURE       2736
NO_SECTION_HEADING|NO_LIST_STRUCTURE        159

Missingness in derived variables
                variable  n_missing percent_missing
n_inclusion_listed_items       7934            4.01
n_exclusion_listed_items      10235            5.18
    n_total_listed_items       2895            1.46
    inclusion_word_count       3772            1.91
    exclusion_word_count       5475            2.77
    mean_sentence_length          0            0.00
     flesch_reading_ease          0            0.00


In [35]:
save_csv_atomic(
    processed_df,
    FINAL_PROCESSED_FILE,
)

print(
    f"Final processed dataset saved to: "
    f"{FINAL_PROCESSED_FILE}"
)

print(f"Rows saved: {len(processed_df)}")
print(
    f"Unique NCT IDs saved: "
    f"{processed_df['nct_id'].nunique()}"
)

Final processed dataset saved to: ctg_full_2005_2025/preprocessing/eligibility_features.csv
Rows saved: 197671
Unique NCT IDs saved: 197671


In [36]:
preprocessing_summary = {
    "input_studies": int(len(cohort_df)),
    "processed_studies": int(len(processed_df)),
    "unique_nct_ids": int(
        processed_df["nct_id"].nunique()
    ),
    "preprocessing_exclusions": 0,

    "disease_domains": {
        "Neoplasms": int(
            processed_df["domain_neoplasms"].sum()
        ),
        "Nervous System Diseases": int(
            processed_df[
                "domain_nervous_system_diseases"
            ].sum()
        ),
        "Cardiovascular Diseases": int(
            processed_df[
                "domain_cardiovascular_diseases"
            ].sum()
        ),
        "Metabolic Diseases": int(
            processed_df[
                "domain_metabolic_diseases"
            ].sum()
        ),
        "Respiratory Tract Diseases": int(
            processed_df[
                "domain_respiratory_tract_diseases"
            ].sum()
        ),
        "Infections": int(
            processed_df["domain_infections"].sum()
        ),
    },

    "disease_domain_membership": {
        "single_domain": int(
            (
                processed_df["n_disease_domains"] == 1
            ).sum()
        ),
        "multiple_domains": int(
            (
                processed_df["n_disease_domains"] > 1
            ).sum()
        ),
    },

    "section_recognition": {
        "inclusion_section": int(
            processed_df["has_inclusion_section"].sum()
        ),
        "exclusion_section": int(
            processed_df["has_exclusion_section"].sum()
        ),
        "no_recognized_section_heading": int(
            (~processed_df["section_split_success"]).sum()
        ),
    },

    "list_structure": {
        "listed_item_count_available": int(
            processed_df[
                "listed_item_count_available"
            ].sum()
        ),
        "listed_item_count_unavailable": int(
            (
                ~processed_df[
                    "listed_item_count_available"
                ]
            ).sum()
        ),
    },

    "warnings": {
        (
            str(row["preprocessing_warning"])
            if str(
                row["preprocessing_warning"]
            ).strip()
            else "NONE"
        ): int(row["n_studies"])
        for _, row in warning_counts.iterrows()
    },

    "missingness": {
        str(row["variable"]): {
            "n_missing": int(row["n_missing"]),
            "percent_missing": float(
                row["percent_missing"]
            ),
        }
        for _, row in missingness_summary.iterrows()
    },
}

with open(
    PREPROCESSING_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        preprocessing_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    f"Preprocessing summary saved to: "
    f"{PREPROCESSING_SUMMARY_FILE}"
)

Preprocessing summary saved to: ctg_full_2005_2025/preprocessing/preprocessing_summary.json


In [37]:
saved_df = pd.read_csv(
    FINAL_PROCESSED_FILE,
    encoding="utf-8-sig",
)

with open(
    PREPROCESSING_SUMMARY_FILE,
    "r",
    encoding="utf-8",
) as f:
    saved_summary = json.load(f)


if len(saved_df) != len(processed_df):
    raise RuntimeError(
        "Saved processed dataset row count does not match "
        "the in-memory dataset."
    )

if saved_df["nct_id"].isna().any():
    raise RuntimeError(
        "Missing NCT IDs found after reading back the saved dataset."
    )

if saved_df["nct_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate NCT IDs found after reading back the saved dataset."
    )


saved_ids = (
    saved_df["nct_id"]
    .astype(str)
    .tolist()
)

input_ids = (
    cohort_df["nct_id"]
    .astype(str)
    .tolist()
)

if saved_ids != input_ids:
    raise RuntimeError(
        "Saved processed dataset does not exactly reconcile "
        "with the disease-analysis cohort."
    )


# Disease-domain metadata must exactly match the preprocessing input.
disease_metadata_columns = [
    "disease_domains",
    "n_disease_domains",
    "disease_classification_status",
    *domain_indicator_columns,
]

for column in disease_metadata_columns:

    saved_values = (
        saved_df[column]
        .astype(str)
        .tolist()
    )

    input_values = (
        cohort_df[column]
        .astype(str)
        .tolist()
    )

    if saved_values != input_values:
        raise RuntimeError(
            f"Saved {column} values do not exactly match "
            "the disease-analysis cohort."
        )


if saved_summary["input_studies"] != len(cohort_df):
    raise RuntimeError(
        "Preprocessing summary input-study count is incorrect."
    )

if saved_summary["processed_studies"] != len(saved_df):
    raise RuntimeError(
        "Preprocessing summary processed-study count is incorrect."
    )

if saved_summary["unique_nct_ids"] != saved_df["nct_id"].nunique():
    raise RuntimeError(
        "Preprocessing summary unique-NCT-ID count is incorrect."
    )

if saved_summary["preprocessing_exclusions"] != 0:
    raise RuntimeError(
        "Preprocessing summary incorrectly reports "
        "preprocessing exclusions."
    )


print(
    "Saved preprocessing outputs passed "
    "read-back integrity checks."
)
print(f"Rows: {len(saved_df)}")
print(
    f"Unique NCT IDs: "
    f"{saved_df['nct_id'].nunique()}"
)
print(
    "Studies with single-domain membership: "
    f"{int((saved_df['n_disease_domains'] == 1).sum())}"
)
print(
    "Studies with multiple-domain membership: "
    f"{int((saved_df['n_disease_domains'] > 1).sum())}"
)
print(
    "Disease domains represented: "
    f"{sum(saved_df[column].any() for column in domain_indicator_columns)}"
)
print(
    f"Preprocessing exclusions: "
    f"{saved_summary['preprocessing_exclusions']}"
)

Saved preprocessing outputs passed read-back integrity checks.
Rows: 197671
Unique NCT IDs: 197671
Studies with single-domain membership: 153677
Studies with multiple-domain membership: 43994
Disease domains represented: 6
Preprocessing exclusions: 0
